In [ ]:
# The abbreviation "OWWW" is C8-WWWDK and "WWNW" is C8-WWNWK

import MDAnalysis as mda
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

OUTPUT_DIR = r".\data\output-dihedral-phi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MANUAL DIHEDRAL DEFINITIONS
# These atom names apply to ALL residues except Octanoyl in each molecule and determine the dihedral "Phi"
DIHEDRAL_DEFINITIONS = {
    'OWWW': {
        1: ['C2', 'C3', 'N4', 'C12'],        # Dihedral 1 
        2: ['C12', 'C13', 'N6', 'C23'],     # Dihedral 2 
        3: ['C23', 'C24', 'N8', 'C34'],     # Dihedral 3 
        4: ['C34', 'C35', 'N9', 'C38'],     # Dihedral 4 
    },
    'WWNW': {
        1: ['C2', 'C3', 'N4', 'C12'],        # Dihedral 1 
        2: ['C12', 'C13', 'N6', 'C23'],     # Dihedral 2 
        3: ['C23', 'C24', 'N8', 'C27'],     # Dihedral 3 
        4: ['C27', 'C28', 'N10', 'C38'],     # Dihedral 4 
    }
}


# Load trajectory
print("Loading trajectory...")
u = mda.Universe(r".\data\mix.tpr", 
                 r".\data\mix.xtc")

print(f"✓ Total frames: {len(u.trajectory)}")
u.trajectory[-1]  # Jump to last frame
print(f"✓ Analyzing LAST frame: {u.trajectory.frame}\n")

# ============================================================================
# DIHEDRAL ANGLE CALCULATION
# ============================================================================

def calculate_dihedral(p1, p2, p3, p4):
    """Calculate dihedral angle between 4 points"""
    try:
        v0 = p1 - p2
        v1 = p3 - p2
        v2 = p4 - p3
        
        n1 = np.cross(v0, v1)
        n2 = np.cross(v1, v2)
        
        n1_norm = n1 / (np.linalg.norm(n1) + 1e-10)
        n2_norm = n2 / (np.linalg.norm(n2) + 1e-10)
        
        x = np.dot(n1_norm, n2_norm)
        y = np.dot(np.cross(n1_norm, n2_norm), v1 / (np.linalg.norm(v1) + 1e-10))
        
        angle = np.degrees(np.arctan2(y, x))
        return angle
    except:
        return np.nan

# ============================================================================
# SELECT MOLECULES
# ============================================================================

print("Selecting molecules...")

mol1_sel = "resname OWWW and not resname SOL"
mol2_sel = "resname WWNW and not resname SOL"

mol1 = u.select_atoms(mol1_sel)
mol2 = u.select_atoms(mol2_sel)

print(f"OWWW atoms: {mol1.n_atoms}")
print(f"WWNW atoms: {mol2.n_atoms}\n")

mol1_residues = mol1.residues
mol2_residues = mol2.residues

print(f"OWWW residues: {mol1_residues.n_residues}")
print(f"WWNW residues: {mol2_residues.n_residues}\n")

# ============================================================================
# CALCULATE DIHEDRALS USING MANUAL MAPPING
# ============================================================================

print("="*100)
print("CALCULATING DIHEDRALS USING MANUAL ATOM MAPPING")
print("="*100 + "\n")

all_dihedrals = []
errors_log = []

# Process OWWW
print("Processing OWWW molecules...")
for residue in mol1_residues:
    res_num = residue.resnum
    
    for dihedral_num, atom_names in DIHEDRAL_DEFINITIONS['OWWW'].items():
        try:
            # Select the 4 atoms
            atom1 = residue.atoms.select_atoms(f"name {atom_names[0]}")
            atom2 = residue.atoms.select_atoms(f"name {atom_names[1]}")
            atom3 = residue.atoms.select_atoms(f"name {atom_names[2]}")
            atom4 = residue.atoms.select_atoms(f"name {atom_names[3]}")
            
            if len(atom1) == 0 or len(atom2) == 0 or len(atom3) == 0 or len(atom4) == 0:
                error_msg = f"OWWW residue {res_num}, dihedral {dihedral_num}: Missing atoms {atom_names}"
                errors_log.append(error_msg)
                continue
            
            p1 = atom1[0].position
            p2 = atom2[0].position
            p3 = atom3[0].position
            p4 = atom4[0].position
            
            dihedral_angle = calculate_dihedral(p1, p2, p3, p4)
            
            # Calculate bond distances
            dist_12 = np.linalg.norm(p2 - p1)
            dist_23 = np.linalg.norm(p3 - p2)
            dist_34 = np.linalg.norm(p4 - p3)
            
            all_dihedrals.append({
                'Molecule': 'OWWW',
                'Residue_Number': res_num,
                'Dihedral_Number': dihedral_num,
                'Atom_1': atom_names[0],
                'Atom_2': atom_names[1],
                'Atom_3': atom_names[2],
                'Atom_4': atom_names[3],
                'Dihedral_Angle': dihedral_angle,
                'Absolute_Angle': abs(dihedral_angle) if not np.isnan(dihedral_angle) else np.nan,
                'Bond_1_2': dist_12,
                'Bond_2_3': dist_23,
                'Bond_3_4': dist_34,
                'Avg_Bond_Distance': np.mean([dist_12, dist_23, dist_34])
            })
            
        except Exception as e:
            error_msg = f"OWWW residue {res_num}, dihedral {dihedral_num}: {str(e)}"
            errors_log.append(error_msg)
            continue

print(f"✓ OWWW dihedrals calculated: {len([d for d in all_dihedrals if d['Molecule'] == 'OWWW'])}\n")

# Process WWNW
print("Processing WWNW molecules...")
for residue in mol2_residues:
    res_num = residue.resnum
    
    for dihedral_num, atom_names in DIHEDRAL_DEFINITIONS['WWNW'].items():
        try:
            atom1 = residue.atoms.select_atoms(f"name {atom_names[0]}")
            atom2 = residue.atoms.select_atoms(f"name {atom_names[1]}")
            atom3 = residue.atoms.select_atoms(f"name {atom_names[2]}")
            atom4 = residue.atoms.select_atoms(f"name {atom_names[3]}")
            
            if len(atom1) == 0 or len(atom2) == 0 or len(atom3) == 0 or len(atom4) == 0:
                error_msg = f"WWNW residue {res_num}, dihedral {dihedral_num}: Missing atoms {atom_names}"
                errors_log.append(error_msg)
                continue
            
            p1 = atom1[0].position
            p2 = atom2[0].position
            p3 = atom3[0].position
            p4 = atom4[0].position
            
            dihedral_angle = calculate_dihedral(p1, p2, p3, p4)
            
            dist_12 = np.linalg.norm(p2 - p1)
            dist_23 = np.linalg.norm(p3 - p2)
            dist_34 = np.linalg.norm(p4 - p3)
            
            all_dihedrals.append({
                'Molecule': 'WWNW',
                'Residue_Number': res_num,
                'Dihedral_Number': dihedral_num,
                'Atom_1': atom_names[0],
                'Atom_2': atom_names[1],
                'Atom_3': atom_names[2],
                'Atom_4': atom_names[3],
                'Dihedral_Angle': dihedral_angle,
                'Absolute_Angle': abs(dihedral_angle) if not np.isnan(dihedral_angle) else np.nan,
                'Bond_1_2': dist_12,
                'Bond_2_3': dist_23,
                'Bond_3_4': dist_34,
                'Avg_Bond_Distance': np.mean([dist_12, dist_23, dist_34])
            })
            
        except Exception as e:
            error_msg = f"WWNW residue {res_num}, dihedral {dihedral_num}: {str(e)}"
            errors_log.append(error_msg)
            continue

print(f"✓ WWNW dihedrals calculated: {len([d for d in all_dihedrals if d['Molecule'] == 'WWNW'])}\n")

# Create DataFrames
df_all = pd.DataFrame(all_dihedrals)
df_owww = df_all[df_all['Molecule'] == 'OWWW'].copy()
df_wwnw = df_all[df_all['Molecule'] == 'WWNW'].copy()

print(f"Total dihedrals: {len(df_all)}")
print(f"  OWWW: {len(df_owww)}")
print(f"  WWNW: {len(df_wwnw)}\n")

# ============================================================================
# PRINT ERROR LOG IF ANY
# ============================================================================

if errors_log:
    print("\n⚠️ ERRORS ENCOUNTERED:")
    print("-" * 100)
    for error in errors_log[:10]:  # Print first 10 errors
        print(f"  {error}")
    if len(errors_log) > 10:
        print(f"  ... and {len(errors_log) - 10} more errors")
    print()

# ============================================================================
# SAVE INDIVIDUAL DATA TO EXCEL
# ============================================================================

print("Saving results to Excel files...\n")

# Save all individual data
excel_all = os.path.join(OUTPUT_DIR, "01_dihedral_angles_all_residues.xlsx")
df_all.to_excel(excel_all, index=False, sheet_name='All_Dihedrals')
print(f"✅ Saved: {excel_all}")

if len(df_owww) > 0:
    excel_owww = os.path.join(OUTPUT_DIR, "02_dihedral_angles_OWWW.xlsx")
    df_owww.to_excel(excel_owww, index=False, sheet_name='OWWW_Individual')
    print(f"✅ Saved: {excel_owww}")

if len(df_wwnw) > 0:
    excel_wwnw = os.path.join(OUTPUT_DIR, "03_dihedral_angles_WWNW.xlsx")
    df_wwnw.to_excel(excel_wwnw, index=False, sheet_name='WWNW_Individual')
    print(f"✅ Saved: {excel_wwnw}")

# ============================================================================
# CALCULATE AVERAGES PER DIHEDRAL
# ============================================================================

print("\n" + "="*100)
print("CALCULATING AVERAGED DIHEDRALS")
print("="*100 + "\n")

# Average dihedrals by dihedral number
averaged_data = []

for molecule in ['OWWW', 'WWNW']:
    mol_df = df_all[df_all['Molecule'] == molecule]
    
    for dihedral_num in sorted(mol_df['Dihedral_Number'].unique()):
        dihedral_df = mol_df[mol_df['Dihedral_Number'] == dihedral_num]
        
        angles = dihedral_df['Dihedral_Angle'].dropna()
        
        if len(angles) > 0:
            averaged_data.append({
                'Molecule': molecule,
                'Dihedral_Number': dihedral_num,
                'Atom_Pair': f"{dihedral_df.iloc[0]['Atom_1']}-{dihedral_df.iloc[0]['Atom_2']}-{dihedral_df.iloc[0]['Atom_3']}-{dihedral_df.iloc[0]['Atom_4']}",
                'Count': len(angles),
                'Mean_Angle': angles.mean(),
                'Median_Angle': angles.median(),
                'Std_Dev': angles.std(),
                'Min_Angle': angles.min(),
                'Max_Angle': angles.max(),
                'Avg_Bond_Distance': dihedral_df['Avg_Bond_Distance'].mean()
            })

df_averaged = pd.DataFrame(averaged_data)

# Save averaged data
excel_avg = os.path.join(OUTPUT_DIR, "04_dihedral_angles_AVERAGED.xlsx")
df_averaged.to_excel(excel_avg, index=False, sheet_name='Averaged_Data')
print(f"✅ Saved: {excel_avg}")

# ============================================================================
# CREATE COMPREHENSIVE SUMMARY EXCEL FILE
# ============================================================================

print("\nCreating comprehensive summary file...\n")

excel_summary = os.path.join(OUTPUT_DIR, "05_SUMMARY_all_analysis.xlsx")

with pd.ExcelWriter(excel_summary, engine='openpyxl') as writer:
    # Sheet 1: Individual OWWW data
    if len(df_owww) > 0:
        df_owww.to_excel(writer, sheet_name='OWWW_Individual', index=False)
    
    # Sheet 2: Individual WWNW data
    if len(df_wwnw) > 0:
        df_wwnw.to_excel(writer, sheet_name='WWNW_Individual', index=False)
    
    # Sheet 3: Averaged data
    df_averaged.to_excel(writer, sheet_name='Averaged_By_Dihedral', index=False)
    
    # Sheet 4: Summary statistics
    summary_stats = []
    
    # Overall statistics
    summary_stats.append(['OVERALL STATISTICS', '', '', '', '', ''])
    summary_stats.append(['', '', '', '', '', ''])
    
    summary_stats.append(['Total OWWW Residues', len(df_owww.groupby('Residue_Number')) if len(df_owww) > 0 else 0, '', '', '', ''])
    summary_stats.append(['Total WWNW Residues', len(df_wwnw.groupby('Residue_Number')) if len(df_wwnw) > 0 else 0, '', '', '', ''])
    summary_stats.append(['Dihedrals per Residue', len(DIHEDRAL_DEFINITIONS['OWWW']), '', '', '', ''])
    summary_stats.append(['Total OWWW Dihedrals', len(df_owww), '', '', '', ''])
    summary_stats.append(['Total WWNW Dihedrals', len(df_wwnw), '', '', '', ''])
    summary_stats.append(['', '', '', '', '', ''])
    
    # OWWW statistics
    summary_stats.append(['OWWW STATISTICS', '', '', '', '', ''])
    for dihedral_num in sorted(df_averaged[df_averaged['Molecule'] == 'OWWW']['Dihedral_Number'].unique()):
        dih_data = df_averaged[(df_averaged['Molecule'] == 'OWWW') & (df_averaged['Dihedral_Number'] == dihedral_num)].iloc[0]
        summary_stats.append([
            f"  Dihedral {dihedral_num}",
            f"Mean: {dih_data['Mean_Angle']:.2f}°",
            f"Std: {dih_data['Std_Dev']:.2f}°",
            f"Min: {dih_data['Min_Angle']:.2f}°",
            f"Max: {dih_data['Max_Angle']:.2f}°",
            f"N={dih_data['Count']}"
        ])
    
    summary_stats.append(['', '', '', '', '', ''])
    
    # WWNW statistics
    summary_stats.append(['WWNW STATISTICS', '', '', '', '', ''])
    for dihedral_num in sorted(df_averaged[df_averaged['Molecule'] == 'WWNW']['Dihedral_Number'].unique()):
        dih_data = df_averaged[(df_averaged['Molecule'] == 'WWNW') & (df_averaged['Dihedral_Number'] == dihedral_num)].iloc[0]
        summary_stats.append([
            f"  Dihedral {dihedral_num}",
            f"Mean: {dih_data['Mean_Angle']:.2f}°",
            f"Std: {dih_data['Std_Dev']:.2f}°",
            f"Min: {dih_data['Min_Angle']:.2f}°",
            f"Max: {dih_data['Max_Angle']:.2f}°",
            f"N={dih_data['Count']}"
        ])
    
    df_summary_stats = pd.DataFrame(summary_stats, columns=['Metric', 'Value', 'Value2', 'Value3', 'Value4', 'Value5'])
    df_summary_stats.to_excel(writer, sheet_name='Summary_Statistics', index=False)

print(f"✅ Saved: {excel_summary}")

# ============================================================================
# PRINT DETAILED SUMMARY
# ============================================================================

print("\n" + "="*100)
print("INDIVIDUAL DATA - OWWW MOLECULE")
print("="*100)

if len(df_owww) > 0:
    print(df_owww[['Residue_Number', 'Dihedral_Number', 'Atom_1', 'Atom_2', 'Atom_3', 'Atom_4', 'Dihedral_Angle']].to_string(index=False))
else:
    print("No data available")

print("\n" + "="*100)
print("INDIVIDUAL DATA - WWNW MOLECULE")
print("="*100)

if len(df_wwnw) > 0:
    print(df_wwnw[['Residue_Number', 'Dihedral_Number', 'Atom_1', 'Atom_2', 'Atom_3', 'Atom_4', 'Dihedral_Angle']].to_string(index=False))
else:
    print("No data available")

print("\n" + "="*100)
print("AVERAGED DIHEDRALS (BY DIHEDRAL NUMBER)")
print("="*100)

print("\nOWWW AVERAGED DIHEDRALS:")
if len(df_averaged[df_averaged['Molecule'] == 'OWWW']) > 0:
    print(df_averaged[df_averaged['Molecule'] == 'OWWW'][['Dihedral_Number', 'Atom_Pair', 'Count', 'Mean_Angle', 'Std_Dev', 'Min_Angle', 'Max_Angle']].to_string(index=False))
else:
    print("No data available")

print("\n\nWWNW AVERAGED DIHEDRALS:")
if len(df_averaged[df_averaged['Molecule'] == 'WWNW']) > 0:
    print(df_averaged[df_averaged['Molecule'] == 'WWNW'][['Dihedral_Number', 'Atom_Pair', 'Count', 'Mean_Angle', 'Std_Dev', 'Min_Angle', 'Max_Angle']].to_string(index=False))
else:
    print("No data available")

# ============================================================================
# PLOTS
# ============================================================================

print("\n" + "="*100)
print("CREATING PLOTS")
print("="*100 + "\n")

# Plot 1: Individual values for each residue by dihedral
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, dihedral_num in enumerate(sorted(DIHEDRAL_DEFINITIONS['OWWW'].keys())):
    ax = axes[idx % 4]
    
    owww_data = df_owww[df_owww['Dihedral_Number'] == dihedral_num]
    wwnw_data = df_wwnw[df_wwnw['Dihedral_Number'] == dihedral_num]
    
    if len(owww_data) > 0:
        ax.scatter(owww_data['Residue_Number'], owww_data['Dihedral_Angle'], 
                  alpha=0.6, s=60, color='steelblue', label='OWWW', edgecolors='black', linewidth=0.5)
    
    if len(wwnw_data) > 0:
        ax.scatter(wwnw_data['Residue_Number'], wwnw_data['Dihedral_Angle'], 
                  alpha=0.6, s=60, color='coral', label='WWNW', edgecolors='black', linewidth=0.5)
    
    ax.axhline(y=0, color='r', linestyle='--', linewidth=1, alpha=0.3)
    ax.set_xlabel('Residue Number', fontsize=10, fontweight='bold')
    ax.set_ylabel('Dihedral Angle (°)', fontsize=10, fontweight='bold')
    ax.set_title(f'Dihedral {dihedral_num}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-180, 180)
    ax.legend(fontsize=9)

plt.tight_layout()
plot_file1 = os.path.join(OUTPUT_DIR, "06_dihedrals_by_residue.png")
plt.savefig(plot_file1, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file1}")
plt.close()

# Plot 2: Averaged dihedrals comparison
fig, ax = plt.subplots(figsize=(12, 6))

owww_avg = df_averaged[df_averaged['Molecule'] == 'OWWW'].sort_values('Dihedral_Number')
wwnw_avg = df_averaged[df_averaged['Molecule'] == 'WWNW'].sort_values('Dihedral_Number')

x = np.arange(len(DIHEDRAL_DEFINITIONS['OWWW']))
width = 0.35

ax.bar(x - width/2, owww_avg['Mean_Angle'], width, label='OWWW', color='steelblue', edgecolor='black', linewidth=1.5)
ax.bar(x + width/2, wwnw_avg['Mean_Angle'], width, label='WWNW', color='coral', edgecolor='black', linewidth=1.5)

# Add error bars
ax.errorbar(x - width/2, owww_avg['Mean_Angle'], yerr=owww_avg['Std_Dev'], 
           fmt='none', ecolor='darkblue', capsize=5, alpha=0.7, linewidth=2)
ax.errorbar(x + width/2, wwnw_avg['Mean_Angle'], yerr=wwnw_avg['Std_Dev'], 
           fmt='none', ecolor='darkred', capsize=5, alpha=0.7, linewidth=2)

ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
ax.set_xlabel('Dihedral Number', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Dihedral Angle (°)', fontsize=12, fontweight='bold')
ax.set_title('Averaged Backbone Dihedrals: OWWW vs WWNW', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'D{i}' for i in range(1, len(DIHEDRAL_DEFINITIONS['OWWW']) + 1)])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(-180, 180)

plt.tight_layout()
plot_file2 = os.path.join(OUTPUT_DIR, "07_averaged_dihedrals_comparison.png")
plt.savefig(plot_file2, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file2}")
plt.close()